In [ ]:
import os

import cmasher as cm
import matplotlib.colors
import numpy as np
import pandas as pd
import plotly.express as px
import scipy


def cmasher_to_plotly(cmap: matplotlib.colors.ListedColormap, n_colors=256):
    """Convert a cmasher listed colormap to a plotly colorscale."""
    colors = (cmap(np.linspace(0, 1, n_colors)) * 255).astype(int)
    colorscale = [[i / (n_colors - 1), f'rgb({r},{g},{b})'] for i, (r, g, b, a) in enumerate(colors)]
    return colorscale

In [ ]:
folder = r"J:\ctgroup\Edward\DATA\VMI\20250227\xe_2,5W\calibrated"

data = pd.DataFrame()
for file in sorted(os.listdir(folder)):
    if file.endswith('.h5'):
        print(file)
        d = pd.read_hdf(os.path.join(folder, file))
        d['angle'] = float(file.split('_')[0])
        d = d[d['pz'] > 0]
        d['px'] = d['px'] - 0.0015

        # rotate in x-z plane by angle degrees
        angle = np.radians(d['angle'].iloc[0] + 5)
        x_new = d['px'] * np.cos(angle) - d['pz'] * np.sin(angle)
        z_new = d['px'] * np.sin(angle) + d['pz'] * np.cos(angle)
        d['px'] = x_new
        d['pz'] = z_new
        d['py'] = d['py'] + 0.009
        data = pd.concat([data, d], ignore_index=True)
        d['px'] = -d['px']
        # d['py']=-d['py']
        d['pz'] = -d['pz']
        data = pd.concat([data, d], ignore_index=True)

data['pr'] = np.sqrt(data['px'] ** 2 + data['py'] ** 2 + data['pz'] ** 2)

data = data[data['pr'] < 0.8]

In [ ]:
(px.density_heatmap(data[data['angle'] == 2], x='px', y='pz', nbinsx=1024, nbinsy=1024,
                    title='Momentum Distribution at 0 Degrees', labels={'px': 'Px (a.u.)', 'pz': 'Pz (a.u.)'},
                    width=800, height=600, color_continuous_scale=cmasher_to_plotly(cm.rainforest),
                    ).add_vline(0, line_dash="dash").add_hline(0, line_dash="dash")
 .show())

px.density_heatmap(
        data[(data['angle'] == 2) & (np.abs(data['pz'] < 0.05))], x='px', y='py', nbinsx=1024, nbinsy=1024,
        width=800, height=600, color_continuous_scale=cmasher_to_plotly(cm.rainforest),
).add_vline(
        0, line_dash="dash"
).add_hline(
        0, line_dash="dash"
).update_layout(
        title='Momentum Distribution at 0 Degrees',
        xaxis_title='Px (a.u.)',
        yaxis_title='Py (a.u.)',
).show()

In [ ]:
for l in range(0, 5):
    for m in range(-l, l + 1):
        data[f'Y{l}{m}'] = scipy.special.sph_harm(m, l, np.arctan2(data['pz'], data['px']),
                                                  np.arccos(data['py'] / data['pr']))



In [ ]:
for l in range(0, 5):
    for m in range(-l, l + 1):
        if m == 0:
            data[f'Y{l}{m}_real'] = data[f'Y{l}{m}'].to_numpy().real
        elif m > 0:
            data[f'Y{l}{m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().real
            data[f'Y{l}{-m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().imag

In [ ]:
means = data.groupby('angle').mean().reset_index()
means_count = data.groupby('angle').count().reset_index()
means_std = data.groupby('angle').std().reset_index()
means_err = means_std / means_count ** 0.5

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

for l in range(0, 5):
    for m in range(-l, l + 1):
        fig.add_scatter(
                x=means['angle'], y=means[f'Y{l}{m}_real'], error_y=dict(type='data', array=means_err[f'Y{l}{m}_real']),
                mode='markers+lines', name=f'Y{l}{m}'
        )
fig.update_layout(
        title='Spherical Harmonics vs Angle',
        xaxis_title='Angle (degrees)',
        yaxis_title='Average Spherical Harmonic Value',
        legend_title='Spherical Harmonics',
        template='plotly_white+presentation',
        width=1200, height=800
)
fig.show()
fig.write_html(os.path.join(folder, 'spherical_harmonics_vs_angle.html'))

In [ ]:
y_df = pd.DataFrame()

#Create dataframe indexed by angle, l, m, with value, err columns

for l in range(0, 5):
    for m in range(-l, l + 1):
        temp_df = pd.DataFrame()
        temp_df['angle'] = means['angle'] - 2
        temp_df['l'] = l
        temp_df['m'] = m
        temp_df['value'] = means[f'Y{l}{m}_real']
        temp_df['err'] = means_err[f'Y{l}{m}_real']
        y_df = pd.concat([y_df, temp_df], ignore_index=True)


def enantiosensitive(row):
    return row['l'] % 2 == 1


def dichroic(row):
    return (row['l'] % 2 == 0 and row['m'] < 0) or (row['l'] % 2 == 1 and row['m'] >= 0)


def allowed(row):
    return row['m'] % 2 == 0


y_df['enantiosensitive'] = y_df.apply(enantiosensitive, axis=1)
y_df['dichroic'] = y_df.apply(dichroic, axis=1)
y_df['allowed'] = y_df.apply(allowed, axis=1)

In [ ]:

def sum_sq(group):
    return np.sqrt(np.sum(group ** 2))


summary = y_df.groupby(['angle', 'l']).agg({'value': sum_sq, 'err': sum_sq,
                                            'enantiosensitive': 'max', 'dichroic': 'max',
                                            'allowed': 'max'}).reset_index()

px.line(
        summary,
        x='l', y='value', color='angle', error_y='err',
        title='Spherical Harmonic Magnitudes vs l',
        labels={'l': 'l', 'value': 'Magnitude', 'angle': 'Angle (degrees)'},
        log_y=True,
)

In [ ]:
px.line(
        y_df[y_df['allowed']][y_df['l'] > 0],
        x='angle', y='value', error_y='err', facet_row='enantiosensitive',
        facet_col='dichroic', color='l', line_dash='m',
        width=1200, height=800, template='plotly_white+presentation',
        title='Spherical Harmonics vs Angle',
).show()
xe_df = y_df.copy()

In [ ]:
folder = r"J:\ctgroup\Edward\DATA\VMI\20250303\Propylene Oxide 2W_calibrated"

data = pd.DataFrame()
for file in sorted(os.listdir(folder)):
    if file.endswith('.h5'):
        print(file)
        d = pd.read_hdf(os.path.join(folder, file))
        d['angle'] = float(file.split('_')[0])
        d = d[d['pz'] > 0]
        d = d[(d['raw_t'] > 200) & (d['raw_t'] < 240)]
        d['px'] = d['px'] - 0.0015

        # rotate in x-z plane by angle degrees
        angle = np.radians(d['angle'].iloc[0] + 5)
        x_new = d['px'] * np.cos(angle) - d['pz'] * np.sin(angle)
        z_new = d['px'] * np.sin(angle) + d['pz'] * np.cos(angle)
        d['px'] = x_new
        d['pz'] = z_new
        d['py'] = d['py'] + 0.009
        data = pd.concat([data, d], ignore_index=True)
        d['px'] = -d['px']
        # d['py']=-d['py']
        d['pz'] = -d['pz']
        data = pd.concat([data, d], ignore_index=True)

data['pr'] = np.sqrt(data['px'] ** 2 + data['py'] ** 2 + data['pz'] ** 2)

data = data[data['pr'] < 0.8]

In [ ]:
(px.density_heatmap(data[data['angle'] == 0], x='px', y='pz', nbinsx=1024, nbinsy=1024,
                    title='Momentum Distribution at 0 Degrees', labels={'px': 'Px (a.u.)', 'pz': 'Pz (a.u.)'},
                    width=800, height=600, color_continuous_scale=cmasher_to_plotly(cm.rainforest),
                    ).add_vline(0, line_dash="dash").add_hline(0, line_dash="dash")
 .show())

px.density_heatmap(
        data[(data['angle'] == 0) & (np.abs(data['pz'] < 0.05))], x='px', y='py', nbinsx=1024, nbinsy=1024,
        width=800, height=600, color_continuous_scale=cmasher_to_plotly(cm.rainforest),
).add_vline(
        0, line_dash="dash"
).add_hline(
        0, line_dash="dash"
).update_layout(
        title='Momentum Distribution at 0 Degrees',
        xaxis_title='Px (a.u.)',
        yaxis_title='Py (a.u.)',
).show()

In [ ]:
for l in range(0, 5):
    for m in range(-l, l + 1):
        data[f'Y{l}{m}'] = scipy.special.sph_harm(m, l, np.arctan2(data['pz'], data['px']),
                                                  np.arccos(data['py'] / data['pr']))
for l in range(0, 5):
    for m in range(-l, l + 1):
        if m == 0:
            data[f'Y{l}{m}_real'] = data[f'Y{l}{m}'].to_numpy().real
        elif m > 0:
            data[f'Y{l}{m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().real
            data[f'Y{l}{-m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().imag

means = data.groupby('angle').mean().reset_index()
means_count = data.groupby('angle').count().reset_index()
means_std = data.groupby('angle').std().reset_index()
means_err = means_std / means_count ** 0.5